# Fraud risk — exploratory data analysis

**Execution status: not executed.** This notebook is prepared locally for Databricks Python compute with Spark, pandas and matplotlib. Empty outputs are intentional. Historical measurements below are attributed to existing reports, not presented as notebook execution results.

## Objective
Establish data quality, class prevalence, and a defensible V1 feature contract before training. Each section separates the question, computation, interpretation, and decision. No model is trained here.

## Read-only boundary
All remote operations read a pinned Delta snapshot. No table/view creation, DML, storage writes, model registration, MLflow runs, package installation, or resource provisioning. Intermediate DataFrames are session objects. Only bounded **aggregates** reach pandas; customer records are never displayed or collected.

Importing this notebook into Databricks creates a workspace resource and requires the user's approval. Execution requires compatible existing compute and read permissions; a SQL warehouse alone does not establish Python notebook compatibility.

## Evidence
- [Phase 0 report](../reports/2026-09-28/profile_report.md)
- [Phase 1 report](../reports/2026-09-28/features_report.md)
- [Data relationships](../data_map_and_features.md)
- [Model requirements](../spec/28-09-26-fraud-model/requirements.md)

Relative links assume the repository is available in the workspace. Source: organizer-supplied synthetic banking data, not production customer outcomes.

In [ ]:
from datetime import datetime, timezone
import re
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

SOURCE_TABLE = "workspace.bank_silver.transactions"
DELTA_VERSION = 1
TRAIN_END = "2025-07-01"
VALIDATION_END = "2026-01-01"
MAX_AGGREGATE_ROWS = 500

assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", SOURCE_TABLE)
assert isinstance(DELTA_VERSION, int) and DELTA_VERSION >= 0
# Do not alter session timezone: record it and interpret boundaries accordingly.
run_context = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": SOURCE_TABLE,
    "delta_version": DELTA_VERSION,
    "spark_version": spark.version,
    "session_timezone": spark.conf.get("spark.sql.session.timeZone"),
    "scope": "full snapshot integrity; train-only feature exploration",
}
display(pd.DataFrame([run_context]))

# No fallback to the latest version if this snapshot is unavailable.
tx = spark.read.option("versionAsOf", DELTA_VERSION).table(SOURCE_TABLE)
required = {
    "transaction_id", "transaction_date", "is_fraud", "amount", "currency",
    "amount_usd", "transaction_type", "channel", "transaction_country",
    "merchant_category", "fraud_score",
}
missing = required - set(tx.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"
tx = tx.select(*sorted(required))

def aggregate_pdf(aggregated):
    """Call only with aggregate/metadata DataFrames; fail on excessive size."""
    result = aggregated.limit(MAX_AGGREGATE_ROWS + 1).toPandas()
    if len(result) > MAX_AGGREGATE_ROWS:
        raise ValueError("Aggregate too large: narrow grouping; do not silently truncate.")
    return result

def n_if(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

def metrics_by(frame, dimension):
    return (frame.groupBy(dimension).agg(
        F.count("*").alias("rows"),
        n_if(F.col("is_fraud") == True).alias("fraud"),
        n_if(F.col("is_fraud").isNull()).alias("null_labels"),
    ).withColumn("labeled_rows", F.col("rows") - F.col("null_labels"))
     .withColumn("fraud_rate_pct", F.when(F.col("labeled_rows") > 0,
                  100.0 * F.col("fraud") / F.col("labeled_rows"))))

## 1. Snapshot schema and integrity
**Question:** Does this snapshot match the documented population, and are keys/labels usable?

The dictionary estimates 5,000,000 transactions. The earlier measured report records 4,425,008 rows and 4,316 positives at Delta v1. Recompute rather than assume agreement. Inspect schema metadata only; do not display transaction rows.

In [ ]:
display(pd.DataFrame([
    {"column": field.name, "type": field.dataType.simpleString(), "nullable": field.nullable}
    for field in tx.schema.fields
]))
summary = aggregate_pdf(tx.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("distinct_ids"),
    n_if(F.col("transaction_id").isNull()).alias("null_ids"),
    n_if(F.col("is_fraud").isNull()).alias("null_labels"),
    n_if(F.col("is_fraud") == True).alias("fraud"),
    n_if(F.col("transaction_date").isNull()).alias("null_dates"),
    F.min("transaction_date").alias("first_transaction"),
    F.max("transaction_date").alias("last_transaction"),
))
s = summary.iloc[0]
total, positives = int(s["rows"]), int(s["fraud"])
assert total > 0, "Empty source snapshot"
duplicate_excess = total - int(s["null_ids"]) - int(s["distinct_ids"])
labeled = total - int(s["null_labels"])
assert labeled > 0, "No labeled transactions"
display(summary)
display(pd.DataFrame([
    {"measurement": "Dictionary estimate", "rows": 5_000_000},
    {"measurement": "Previous team report", "rows": 4_425_008},
    {"measurement": "Current pinned snapshot", "rows": total},
]))
print(f"Duplicate non-null ID excess: {duplicate_excess:,}")
print(f"Measured fraud prevalence: {100 * positives / labeled:.4f}%")
print(f"Always-negative accuracy: {100 * (labeled - positives) / labeled:.4f}%")
assert duplicate_excess == 0 and int(s["null_ids"]) == 0, "Resolve key quality before modeling"
assert int(s["null_labels"]) == 0 and int(s["null_dates"]) == 0, "Define exclusions before modeling"

**Interpretation / decision:** passing these checks establishes structural integrity, not label correctness or learnable signal. A high always-negative accuracy is expected with rare fraud and is not evidence of useful detection.

**After execution:** record discrepancies and their provenance here. Do not assume the dictionary shortfall means failed ingestion without checking delivery evidence.

## 2. Temporal partition viability — full snapshot
**Question:** Do all proposed periods contain both classes?

Only counts, date ranges, and positive counts are inspected across test. Feature/label exploration below uses **train only**. Existing Phase 0 reports already expose aggregate test statistics; retain that disclosure and avoid further test-driven feature selection. Boundaries use the recorded Spark session timezone, whose relation to source timestamps remains unconfirmed.

In [ ]:
partitioned = tx.withColumn("split", F.when(
    F.col("transaction_date").isNull(), "invalid_date"
).when(F.col("transaction_date") < F.lit(TRAIN_END).cast("timestamp"), "train")
 .when(F.col("transaction_date") < F.lit(VALIDATION_END).cast("timestamp"), "validation")
 .otherwise("test"))
splits = aggregate_pdf(metrics_by(partitioned, "split").orderBy("split"))
display(splits)
assert set(splits["split"]) == {"train", "validation", "test"}, "Unexpected/absent partition"
assert ((splits["fraud"] > 0) & (splits["fraud"] < splits["labeled_rows"])).all(), "Both classes required"
train = partitioned.filter(F.col("split") == "train")
monthly = aggregate_pdf(metrics_by(
    train.withColumn("month", F.date_format("transaction_date", "yyyy-MM")), "month"
).orderBy("month"))
display(monthly)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].bar(monthly["month"], monthly["rows"], color="#2563eb")
axes[0].set_ylabel("Transactions")
axes[1].plot(monthly["month"], monthly["fraud_rate_pct"].astype(float), marker="o", color="#b45309")
axes[1].set_ylabel("Fraud (%)")
axes[1].tick_params(axis="x", rotation=75)
fig.suptitle("Training period only — volume and fraud prevalence")
fig.tight_layout()
plt.show()

**Interpretation / decision:** positive counts establish feasibility, not statistical precision. Compare monthly changes with their sample sizes. Partial boundary months must not be mistaken for demand drops. Reconcile timezone before freezing training splits; no model selection has occurred.

## 3. Candidate feature coverage — train only
**Question:** Which fields need explicit missing-value handling?

Measure both nulls and blank category strings. Missingness alone does not justify dropping a feature; its usefulness must later be tested on validation.

In [ ]:
feature_columns = ["amount", "currency", "amount_usd", "transaction_type", "channel",
                   "transaction_country", "merchant_category", "transaction_date"]
cat_columns = ["currency", "transaction_type", "channel", "transaction_country", "merchant_category"]
expressions = [F.count("*").alias("rows")]
expressions += [n_if(F.col(c).isNull()).alias(c + "_nulls") for c in feature_columns]
expressions += [n_if(F.trim(F.col(c)) == "").alias(c + "_blanks") for c in cat_columns]
coverage_row = aggregate_pdf(train.agg(*expressions)).iloc[0]
coverage = pd.DataFrame([
    {"feature": c, "nulls": int(coverage_row[c + "_nulls"]),
     "null_pct": 100 * int(coverage_row[c + "_nulls"]) / int(coverage_row["rows"]),
     "blank_strings": int(coverage_row[c + "_blanks"]) if c in cat_columns else None}
    for c in feature_columns
])
display(coverage)
coverage.plot.barh(x="feature", y="null_pct", legend=False, color="#2563eb", figsize=(8, 4))
plt.xlabel("Missing values (%) — train only")
plt.tight_layout()
plt.show()

**Prior evidence, not this cell's output:** full-snapshot Phase 0 reported approximately 76.8% missing merchant_category. Phase 1 retained it as a candidate with explicit missing representation. Imputers, encoders, and scalers remain unfitted and must learn from train only.

## 4. Currency normalization — train only
**Question:** Can USD-native amounts fill conversion gaps without mixing currencies?

Candidate: `CASE WHEN currency = 'USD' THEN amount ELSE amount_usd END`. Never substitute non-USD raw amounts for missing conversions. No exchange-rate table is joined or modified. The full-snapshot Phase 1 report found 99,477 normalized nulls (2.25%); train-only counts below will differ.

In [ ]:
normalized = train.withColumn("amount_usd_norm", F.when(
    F.col("currency") == "USD", F.col("amount")
).otherwise(F.col("amount_usd")))
fx = aggregate_pdf(normalized.groupBy("currency").agg(
    F.count("*").alias("rows"),
    n_if(F.col("amount").isNull()).alias("missing_amount"),
    n_if(F.col("amount_usd").isNull()).alias("missing_raw_usd"),
    n_if(F.col("amount_usd_norm").isNull()).alias("missing_normalized_usd"),
    n_if(F.col("amount") <= 0).alias("nonpositive_amount"),
    n_if((F.col("currency") == "USD") & F.col("amount_usd").isNotNull()
         & (F.abs(F.col("amount") - F.col("amount_usd")) > 0.01)).alias("usd_disagreement"),
    F.min("amount").alias("min_amount"), F.max("amount").alias("max_amount"),
).orderBy("currency"))
display(fx)
ratios = aggregate_pdf(normalized.filter(
    (F.col("currency") != "USD") & (F.col("amount") > 0) & F.col("amount_usd").isNotNull()
).withColumn("ratio", F.col("amount_usd") / F.col("amount"))
 .groupBy("currency").agg(F.count("*").alias("rows"),
     F.min("ratio").alias("min_ratio"), F.avg("ratio").alias("mean_ratio"),
     F.max("ratio").alias("max_ratio"), n_if(F.col("ratio") <= 0).alias("nonpositive_ratio")))
display(ratios)
amount_distribution = aggregate_pdf(normalized.agg(
    F.expr("percentile_approx(amount_usd_norm, array(0.01, 0.25, 0.5, 0.75, 0.95, 0.99), 10000)").alias("usd_quantiles"),
    F.count("amount_usd_norm").alias("observed_usd_amounts"),
))
display(amount_distribution)

**Interpretation / decision:** internal conversion consistency is not proof of historically correct exchange rates or availability at decision time. Keep unresolved conversion values missing. Retain raw amount with currency and normalized USD as candidates; freeze the policy before model evaluation.

## 5. Exploratory fraud rates — train only
**Question:** Do country, channel, transaction type, and merchant category show differences worth evaluating?

These are unadjusted associations, not causal explanations or evidence that a country/customer is fraudulent. Report positives and denominators beside rates. Sparse groups can produce unstable estimates.

In [ ]:
for column in ["transaction_country", "channel", "transaction_type", "merchant_category"]:
    category = F.when(F.col(column).isNull() | (F.trim(F.col(column)) == ""), "__missing__").otherwise(F.col(column))
    grouped = metrics_by(train.withColumn(column, category), column).orderBy(F.desc("rows"))
    result = aggregate_pdf(grouped)
    display(result)
    result.plot.bar(x=column, y="fraud_rate_pct", legend=False, color="#2563eb", figsize=(10, 3))
    plt.ylabel("Fraud (%) — train only")
    plt.title(f"Unadjusted rates by {column}; inspect table denominators")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

**After execution:** record which differences are supported by adequate positive counts and which are uncertain. Do not select features by inspecting test. Train-only exploratory differences still require held-out validation.

## 6. Suspected leakage diagnostic — train only
**Question:** How closely does the supplied fraud_score track the label?

The full-snapshot report found 999 scores >=70, all labeled fraud. That is perfect precision for this subset, not perfect detection of all 4,316 fraud cases, and not proof of the score's provenance. Keep fraud_score excluded until its source and temporal availability are established.

In [ ]:
diagnostic = train.withColumn("score_band", F.when(F.col("fraud_score").isNull(), "missing")
    .when(F.col("fraud_score") >= 70, ">=70").otherwise("<70"))
score_bands = aggregate_pdf(metrics_by(diagnostic, "score_band").orderBy("score_band"))
display(score_bands)
train_positive_count = int(splits.loc[splits["split"] == "train", "fraud"].iloc[0])
high = score_bands.loc[score_bands["score_band"] == ">=70"]
if not high.empty:
    high_row = high.iloc[0]
    print("Diagnostic subset precision:", float(high_row["fraud"]) / float(high_row["labeled_rows"]) if high_row["labeled_rows"] else None)
    print("Share of training fraud covered:", float(high_row["fraud"]) / train_positive_count if train_positive_count else None)

**Decision:** exclude fraud_score, is_fraud as a predictor, transaction_status, response_code, and processing metadata. A predictive association is not permission to use a feature unavailable at inference. These diagnostics are not model performance.

## 7. V1 feature preparation preview — expressions only
**Question:** Can we define predictors separately from IDs and labels without persisting rows?

This mirrors the documented V1 design. The existing `ml/features.py` remains the implementation reference. The notebook explicitly preserves nulls in sign/weekend derivations and normalizes blank categories. Existing Python and SQL helpers have edge-case differences for missing inputs; align them before production training rather than claiming exact parity. Current source reports contain no null amounts/dates, but that does not replace edge-case tests.

In [ ]:
categorical = ["currency", "transaction_type", "channel", "transaction_country", "merchant_category"]
base = normalized.withColumn("log_abs_amount", F.log1p(F.abs(F.col("amount").cast("double"))))
base = base.withColumn("amount_sign", F.when(F.col("amount").isNull(), F.lit(None).cast("int"))
    .when(F.col("amount") > 0, 1).when(F.col("amount") < 0, -1).otherwise(0))
base = base.withColumn("hour", F.hour("transaction_date"))
base = base.withColumn("weekday", F.pmod(F.dayofweek("transaction_date") + F.lit(5), F.lit(7)))
base = base.withColumn("is_weekend", F.when(F.col("weekday").isNull(), F.lit(None).cast("int"))
    .when(F.col("weekday") >= 5, 1).otherwise(0))
for column in categorical:
    base = base.withColumn(column, F.when(F.col(column).isNull() | (F.trim(F.col(column)) == ""),
                                       "__missing__").otherwise(F.trim(F.col(column))))
predictors = ["amount", *categorical, "log_abs_amount", "amount_sign", "amount_usd_norm", "hour", "weekday", "is_weekend"]
excluded = {"transaction_id", "customer_id", "product_id", "is_fraud", "fraud_score",
            "transaction_status", "response_code", "process_date"}
assert not set(predictors) & excluded
X_preview = base.select(*predictors)  # Lazy, in-memory plan; no row display or persistence.
display(pd.DataFrame([{"predictor": f.name, "type": f.dataType.simpleString()} for f in X_preview.schema.fields]))
print("Predictors:", len(predictors), "| No fitted preprocessing, training, or data writes.")

## 8. Findings and decision record

Complete this table **after running the notebook**, referring to the output of each section. Keep prior measurements distinguishable from this run.

| Question | Prior evidence | This run / interpretation | Decision |
|---|---|---|---|
| Population | 4,425,008 rows at v1 vs 5M dictionary estimate | Pending execution | Reconcile delivery evidence if changed |
| Label balance | 4,316 fraud, approximately 0.0975% | Pending execution | AP/precision/recall, not accuracy alone |
| Split viability | 3,014 / 699 / 603 positives | Pending execution; timezone unresolved | Preserve final test |
| USD normalization | 99,477 missing normalized values, 2.25% overall | Train-only output pending | Do not mix raw currencies |
| Merchant category | Approximately 76.8% missing overall | Train-only output pending | Missing category remains a candidate |
| Fraud score | >=70 identifies 999 positives, zero negatives overall | Train-only output pending; provenance unknown | Exclude from predictors |
| Feature parity | Python/SQL helper null behavior differs | Documented during code review | Align helpers before training |

### Remaining limitations
Synthetic label generation and label maturation are unknown. Source timezone and exchange-rate availability remain unverified. Snapshot dimensions do not establish historical attributes. No fitted model, calibrated probability, or production improvement has been demonstrated.

### Next step
Review findings, reconcile feature edge cases, and confirm training compute. Fit preprocessing on train only, compare B0/B1/M1/M2 on validation, and reserve test for final evaluation. Registering experiments or publishing resources requires separate approval under the current workflow.

### Execution checklist
- [ ] User approved notebook import and use of existing compute.
- [ ] Ran top-to-bottom with no errors on the specified snapshot.
- [ ] Recorded timezone, runtime, execution time, and scope.
- [ ] Filled findings with output-backed observations.
- [ ] Confirmed no individual records or credentials in outputs.
- [ ] Reviewed the executed notebook before saving/sharing outputs.

### Technical references
- [Databricks Delta snapshot reads](https://docs.databricks.com/aws/en/delta/tutorial)
- [Delta history and time-travel availability](https://docs.databricks.com/aws/en/tables/history)
- [Spark toPandas: driver-memory constraint](https://spark.apache.org/docs/3.5.9/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toPandas.html)

## 9. Aggregate run result

Return only bounded aggregate metrics to the one-time run record so they can be reviewed and reported. No transaction-level rows or identifiers are included.

In [ ]:
import json
import math

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if hasattr(value, "tolist"):
        return json_safe(value.tolist())
    if hasattr(value, "item"):
        return json_safe(value.item())
    if value is None or isinstance(value, (str, int, bool)):
        return value
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    return str(value)

def records(frame):
    return json_safe(frame.to_dict(orient="records"))

result_payload = {
    "run_context": run_context,
    "snapshot_metrics": records(summary),
    "split_metrics": records(splits),
    "training_monthly_metrics": records(monthly),
    "training_feature_coverage": records(coverage),
    "training_currency_coverage": records(fx),
    "training_conversion_ratios": records(ratios),
    "training_amount_distribution": records(amount_distribution),
    "training_fraud_score_bands": records(score_bands),
}
dbutils.notebook.exit(json.dumps(result_payload, allow_nan=False))
